# nano-prover SFT on Colab (free T4 GPU)

The Colab kernel runs on a **remote Google GPU machine** (files live under `/content`, not your laptop).
The `sft_train.py` that actually runs is the one **cloned by cell 2** — so whenever the code was updated,
**re-run cell 2** (it force-pulls the latest) before cell 5. **Prereq:** commit + push first. Run top to bottom.

In [ ]:
# 1. confirm we have a GPU (must show Tesla T4 / similar; 'command not found' => CPU runtime, redo as GPU)
!nvidia-smi

In [ ]:
# 2. clone + FORCE to the latest main (re-run this whenever the code changed).  No nesting.
#    Private repo: replace the URL with https://<GITHUB_TOKEN>@github.com/FELIXGAO0315/Conjecture-Gym.git
import os
os.chdir('/content')
if not os.path.isdir('/content/Conjecture-Gym'):
    !git clone --depth 1 https://github.com/FELIXGAO0315/Conjecture-Gym.git
os.chdir('/content/Conjecture-Gym')
!git fetch -q --depth 1 origin main && git reset -q --hard origin/main
assert os.path.exists('output/train_corpus.jsonl'), 'corpus not found — did you push it?'
print('cwd:', os.getcwd(), '| HEAD:', end=' ')
!git log --oneline -1

In [ ]:
# 3. install the SFT stack.  (torchao is handled inside sft_train.py — it removes Colab's old one.)
!pip install -q -U transformers trl peft datasets accelerate

In [ ]:
# 4. build train/test splits from the corpus
!python training/prepare_data.py

In [ ]:
# 5. LoRA SFT — defaults: 1 epoch, max_seq 1024, grad-ckpt off  => ~335 steps, ~30 min on T4.
#    Faster first pass (~10 min): !SFT_MODEL=Qwen/Qwen2.5-Coder-0.5B-Instruct python training/sft_train.py
#    OOM? !SFT_GRAD_CKPT=1 python training/sft_train.py   |   more epochs? SFT_EPOCHS=3
!python training/sft_train.py

## Get the adapter back (`training/out/`), then evaluate LOCALLY
Pick one:

In [ ]:
# OPTION A — zip for manual download (browser Colab: a download prompt; VS Code: find out.zip in the file tree)
!cd training && zip -rq out.zip out && ls -lh out.zip
try:
    from google.colab import files; files.download('training/out.zip')
except Exception as e:
    print('download via the file panel instead:', e)

In [ ]:
# OPTION B — push the adapter back to GitHub (needs a token with repo write).  Then `git pull` locally.
# TOKEN = 'ghp_xxx'   # a GitHub Personal Access Token (classic, repo scope)
# !git config user.email you@example.com && git config user.name you
# !git add -f training/out && git commit -m 'trained adapter'
# !git push https://{TOKEN}@github.com/FELIXGAO0315/Conjecture-Gym.git main

Back on your **local** machine (the Lean verifier lives there):
```bash
git pull                 # if you used Option B
# or unzip the downloaded out.zip into training/out/
python training/eval_lean.py --n 50 --k 1   # real Lean-verified pass@k
```